# Tokens et logits — imports

Charge PyTorch et le modèle du projet.

In [1]:
import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.model import load_model

# Tokenisation

Transforme le texte en tenseurs exploitables par le modèle.

In [2]:
llm = load_model()

tokenizer = llm.tokenizer
model = llm.model
device = llm.device

text = "Paris est la capitale de"

encoded = tokenizer(
    text,
    return_tensors="pt",
)

encoded

Chargement du modèle : Qwen/Qwen2.5-0.5B-Instruct
Device utilisé : cpu


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'input_ids': tensor([[59604,  1788,  1187, 60410,  1574,   409]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

# IDs des tokens

Affiche les identifiants produits par le tokenizer.

In [3]:
input_ids = encoded["input_ids"]

print("Shape :", input_ids.shape)
print("IDs :", input_ids)

Shape : torch.Size([1, 6])
IDs : tensor([[59604,  1788,  1187, 60410,  1574,   409]])


# Tokens réels

Associe chaque identifiant à son token.

In [4]:
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

for token_id, token in zip(input_ids[0].tolist(), tokens):
    print(f"{token_id:>8} -> {repr(token)}")

   59604 -> 'Paris'
    1788 -> 'Ġest'
    1187 -> 'Ġla'
   60410 -> 'Ġcapit'
    1574 -> 'ale'
     409 -> 'Ġde'


# Tokens vs mots

Compare le nombre de tokens au nombre de mots.

In [5]:
print("Nombre de tokens :", len(tokens))
print("Nombre de mots approximatif :", len(text.split()))

Nombre de tokens : 6
Nombre de mots approximatif : 5


# Décodage

Reconstruit le texte à partir des IDs.

In [6]:
decoded = tokenizer.decode(input_ids[0])

print("Original :", text)
print("Décodé  :", decoded)

Original : Paris est la capitale de
Décodé  : Paris est la capitale de


# Exemples de tokenisation

Observe le découpage de plusieurs chaînes.

In [7]:
examples = [
    "chat",
    "anticonstitutionnellement",
    "Transformer",
    "🤖",
]

for example in examples:
    ids = tokenizer.encode(example, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(ids)

    print()
    print("Texte :", example)
    print("IDs   :", ids)
    print("Tokens:", tokens)


Texte : chat
IDs   : [9686]
Tokens: ['chat']

Texte : anticonstitutionnellement
IDs   : [517, 1924, 10446, 8138, 986]
Tokens: ['ant', 'icon', 'stitution', 'nel', 'lement']

Texte : Transformer
IDs   : [46358]
Tokens: ['Transformer']

Texte : 🤖
IDs   : [147238]
Tokens: ['ðŁ¤ĸ']


# Transfert sur le device

Place les tenseurs sur CPU ou GPU.

In [8]:
inputs = {
    key: value.to(device)
    for key, value in encoded.items()
}

# Forward pass

Fait passer l’entrée dans le Transformer.

In [9]:
with torch.no_grad():
    outputs = model(**inputs)

# Logits

Observe la forme de la sortie du modèle.

In [10]:
logits = outputs.logits

print(logits.shape)

torch.Size([1, 6, 151936])


# Dernière position

Récupère les logits utilisés pour prédire le prochain token.

In [11]:
last_token_logits = logits[:, -1, :]

print(last_token_logits.shape)
print(last_token_logits)

torch.Size([1, 151936])
tensor([[ 2.7969,  3.2812,  3.9531,  ..., -4.9062, -4.9062, -4.9062]],
       dtype=torch.bfloat16)


# Softmax

Convertit les logits en probabilités.

In [12]:
probabilities = torch.softmax(
    last_token_logits,
    dim=-1,
)

print(probabilities.shape)
print(probabilities.sum())

torch.Size([1, 151936])
tensor(1., dtype=torch.bfloat16)


# Top 10

Affiche les tokens les plus probables.

In [14]:
top_k = 10

top_probs, top_ids = torch.topk(
    probabilities[0],
    k=top_k,
)

for probability, token_id in zip(
    top_probs.tolist(),
    top_ids.tolist(),
):
    token = tokenizer.decode([token_id])

    print(
        f"{repr(token):20s} "
        f"id={token_id:<8} "
        f"probabilité={probability:.4f}"
    )

' la'                id=1187     probabilité=0.4844
' l'                 id=326      probabilité=0.2754
' France'            id=9625     probabilité=0.0510
' ce'                id=3761     probabilité=0.0146
' qui'               id=7774     probabilité=0.0083
' notre'             id=28349    probabilité=0.0042
' ____'              id=30743    probabilité=0.0037
' __'                id=1304     probabilité=0.0037
' qué'               id=42288    probabilité=0.0035
' ('                 id=320      probabilité=0.0035


# Argmax

Sélectionne le token le plus probable.

In [16]:
next_token_id = torch.argmax(
    probabilities,
    dim=-1,
)

print("ID :", next_token_id.item())

next_token = tokenizer.decode(
    next_token_id
)

print("Token choisi :", repr(next_token))

ID : 1187
Token choisi : ' la'


# Vérification

Confirme que logits et softmax donnent le même argmax.

In [17]:
next_token_from_logits = torch.argmax(
    last_token_logits,
    dim=-1,
)

assert next_token_id.item() == next_token_from_logits.item()

# Ajout d’un token

Ajoute manuellement le token prédit à la séquence.

In [18]:
new_input_ids = torch.cat(
    [
        inputs["input_ids"],
        next_token_id.unsqueeze(0),
    ],
    dim=-1,
)

print(
    tokenizer.decode(
        new_input_ids[0],
        skip_special_tokens=True,
    )
)

Paris est la capitale de la


# Greedy decoding

Teste la génération gloutonne implémentée dans le projet.

In [19]:
from src.llm.decoding import greedy_generate

result = greedy_generate(
    model=model,
    tokenizer=tokenizer,
    prompt="Paris est la capitale de",
    device=device,
    max_new_tokens=20,
)

print(result)

Paris est la capitale de la France, mais elle est aussi la capitale de l'Europe. Elle est la plus grande


# `model.generate()`

Compare avec la génération native de Transformers.

In [20]:
encoded = tokenizer(
    "Paris est la capitale de",
    return_tensors="pt",
).to(device)

with torch.no_grad():

    generated_ids = model.generate(
        **encoded,
        max_new_tokens=20,
        do_sample=False,
    )

print(
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    )
)

Paris est la capitale de l'Inde, mais elle n'est pas le pays le plus grand. Elle est en effet
